# Memory · strategy comparison

FWI backward passes need access to the full forward wavefield to compute the imaging condition. The three common ways to make that tractable are:

- **Full wavefield (no savings):** store every timestep; fastest but most memory-hungry.
- **Boundary saving:** save a stencil-width strip along each side of the physical interior + last two interior snapshots; replay forward by reverse-time propagation during backward. Cheap memory, ~2× more compute.
- **Checkpointing:** save snapshots at intermediate timesteps; replay forward from the nearest checkpoint when backward needs a missing timestep.
- **Boundary dtype compression (NEW):** ``storage_dtype`` ∈ ``{'fp32', 'fp16', 'bf16', 'int8'}`` reduces boundary buffer size with cast/quantize at storage boundary; compute stays FP32.

This notebook runs **one forward + backward step** of a small acoustic FWI problem under different memory strategies and reports peak GPU memory + wallclock + boundary buffer size.

## 1. Setup — one shot on 25 m Marmousi

In [ ]:
import os
import time
import numpy as np
import torch
import matplotlib.pyplot as plt

from sweep.equations import Acoustic
from sweep.propagator.torch import PropTorch
from sweep.signal import ricker
from sweep.datasets import load_marmousi, MARMOUSI_DH

dh, dt, nt = MARMOUSI_DH * 2, 0.002, 2000   # 25 m, 4 s
freq, delay = 5.0, 0.20

vp_true_np = load_marmousi('true')[::2, ::2].copy()
vp_init_np = load_marmousi('smooth')[::2, ::2].copy()
shape = vp_true_np.shape
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, ' shape:', shape, ' record:', f'{nt*dt:.1f} s')

nshots, nrec = 1, 100
src_x = np.array([shape[1] // 2], dtype=np.int64)
rec_x = np.linspace(0, shape[1] - 1, nrec).astype(np.int64)
sources   = np.stack([src_x, np.full(nshots, 2, dtype=np.int64)], axis=1)
receivers = np.stack([rec_x, np.full(nrec, 4, dtype=np.int64)], axis=1)
receivers = np.repeat(receivers[None, ...], nshots, axis=0)
t = np.arange(nt, dtype=np.float32) * dt
wavelet = ricker(t - delay, f=freq)
vp_true_t = torch.tensor(vp_true_np, device=device)


## 2. Generate the observed data once

In [ ]:
with torch.no_grad():
    eq = Acoustic(device=device)
    solver_obs = PropTorch(eq, shape=shape, dh=dh, dt=dt, nt=nt)
    observed = solver_obs(wavelet, sources, receivers, models=[vp_true_t]).detach()
print('observed:', tuple(observed.shape))
del solver_obs
torch.cuda.empty_cache()


### Aside · which sources and receivers does this equation accept?

In [ ]:
print('models (input order matters!):')
for spec in eq.MODEL_SPECS:
    unit = f' [{spec.unit}]' if spec.unit else ''
    print(f'  - {spec.name:8s}{unit}  — {spec.description}')
print('defaults  :', eq.default_source_fields, '/', eq.default_receiver_fields)
print('receivers :')
for spec in eq.available_receiver_fields():
    print(f'  - {spec.name:8s} aliases={spec.aliases}  — {spec.description}')


## 3. The strategies

```
PropTorch
 ├── eager_options : EagerOptions          # eager-only
 └── cuda_options  : CUDAOptions           # c-only memory block
      └── memory   : MemoryOptions         # pick one strategy:
           ├── boundary : BoundaryOptions  # storage / pinning / interval / storage_dtype
           └── ckpt     : CkptOptions      # chunk / recursive
```

In [ ]:
strategies = []

# --- eager backend (top-level kwargs) --------------------------------
strategies.append((
    'eager · full',
    dict(impl='eager', use_ckpt=False),
))
strategies.append((
    'eager · chunk-ckpt',
    dict(impl='eager', use_ckpt=True, ckpt_chunks=100),
))

# --- compiled c backend ---------------------------------------------
strategies.append((
    'c · full',
    dict(impl='c', use_ckpt=False),
))

# memory = MemoryOptions(strategy='boundary', boundary=BoundaryOptions(...))
strategies.append((
    'c · boundary (GPU)',
    dict(impl='c',
         memory={'strategy': 'boundary',
                 'boundary': {'storage': 'gpu'}}),
))

# --- compressed boundary storage ---
# storage_dtype ∈ {'fp32', 'fp16', 'bf16', 'int8'}; cast at storage
# boundary inside the kernel (compute stays FP32).  fp16/bf16 halve the
# boundary buffer; int8 does per-256-cell adaptive quantization
# (~4x compression with FP32-equivalent gradient quality across
# 15 C-bound equations — see test/solver_gradient_mode_suite.py).
strategies.append((
    'c · boundary GPU + fp16',
    dict(impl='c',
         memory={'strategy': 'boundary',
                 'boundary': {'storage': 'gpu',
                              'storage_dtype': 'fp16'}}),
))
strategies.append((
    'c · boundary GPU + bf16',
    dict(impl='c',
         memory={'strategy': 'boundary',
                 'boundary': {'storage': 'gpu',
                              'storage_dtype': 'bf16'}}),
))
strategies.append((
    'c · boundary GPU + int8',
    dict(impl='c',
         memory={'strategy': 'boundary',
                 'boundary': {'storage': 'gpu',
                              'storage_dtype': 'int8'}}),
))

# --- CPU-staged boundary ---
strategies.append((
    'c · boundary (CPU pinned)',
    dict(impl='c',
         memory={'strategy': 'boundary',
                 'boundary': {'storage': 'cpu',
                              'pinned_memory': True}}),
))
strategies.append((
    'c · boundary (CPU, interval=4)',
    dict(impl='c',
         memory={'strategy': 'boundary',
                 'boundary': {'storage': 'cpu',
                              'pinned_memory': True,
                              'transfer_interval': 4}}),
))

# --- checkpointing ---
strategies.append((
    'c · chunk-ckpt',
    dict(impl='c',
         memory={'strategy': 'ckpt',
                 'ckpt': {'mode': 'chunk', 'chunks': 100}}),
))
strategies.append((
    'c · recursive-ckpt',
    dict(impl='c',
         memory={'strategy': 'ckpt',
                 'ckpt': {'mode': 'recursive', 'count': 8}}),
))

for label, kw in strategies:
    print(f'{label:34s} → {kw}')


## 4. Measure peak memory + wallclock + boundary buffer

One forward + one MSE gradient against the smooth start. Three metrics: total GPU peak, boundary buffer size (what ``storage_dtype`` actually controls), wallclock.

In [ ]:
def measure(strategy_label, kwargs):
    os.environ.pop('SWEEP_BOUNDARY_DTYPE', None)
    os.environ.pop('SWEEP_FP16_BOUNDARY', None)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    eq = Acoustic(device=device)
    solver = PropTorch(eq, shape=shape, dh=dh, dt=dt, nt=nt, **kwargs)
    vp_p = torch.nn.Parameter(torch.tensor(vp_init_np, device=device))
    torch.cuda.synchronize(); t0 = time.time()
    pred = solver(wavelet, sources, receivers, models=[vp_p])
    loss = (pred - observed).pow(2).mean()
    loss.backward()
    torch.cuda.synchronize()
    dt_s = time.time() - t0
    peak = torch.cuda.max_memory_allocated() / 2**30
    grad_l2 = vp_p.grad.detach().pow(2).mean().sqrt().item()
    try:
        bg = solver._backend_impl.boundary_gpu_full
        bdy_mib = sum(t.element_size() * t.numel() for t in bg) / 2**20
        bdy_dtype = str(bg[0].dtype).replace('torch.', '') if bg else 'n/a'
    except (AttributeError, TypeError):
        bdy_mib, bdy_dtype = 0.0, 'n/a'
    del solver, vp_p, pred, loss
    torch.cuda.empty_cache()
    return dict(label=strategy_label, time_s=dt_s, peak_gb=peak, grad_l2=grad_l2,
                bdy_mib=bdy_mib, bdy_dtype=bdy_dtype)

results = []
for label, kw in strategies:
    r = measure(label, kw)
    results.append(r)
    print(f"{r['label']:34s} peak={r['peak_gb']*1024:7.1f} MiB  bdy={r['bdy_mib']:6.1f} MiB ({r['bdy_dtype']})  fwd+bwd={r['time_s']:5.2f}s  ||grad||₂={r['grad_l2']:.3e}")


## 5. Visual summary

In [ ]:
labels   = [r['label'] for r in results]
peaks    = [r['peak_gb'] * 1024 for r in results]   # GB → MiB
bdy_mibs = [r['bdy_mib'] for r in results]
times    = [r['time_s'] for r in results]

fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
bars0 = axes[0].barh(labels, peaks, color='steelblue')
axes[0].set_xlabel('peak GPU memory (MiB)')
axes[0].set_title('total GPU peak')
axes[0].bar_label(bars0, fmt='%.0f', padding=3)
axes[0].invert_yaxis()

bars1 = axes[1].barh(labels, bdy_mibs, color='seagreen')
axes[1].set_xlabel('boundary buffer (MiB)')
axes[1].set_title('boundary buffer only')
axes[1].bar_label(bars1, fmt='%.1f', padding=3)
axes[1].invert_yaxis()

bars2 = axes[2].barh(labels, times, color='indianred')
axes[2].set_xlabel('one fwd+bwd pass (s)')
axes[2].set_title('wallclock cost')
axes[2].bar_label(bars2, fmt='%.2f s', padding=3)
axes[2].invert_yaxis()
plt.show()
